In [1]:
import os
import time
import requests
import numpy as np
import rasterio
from rasterio.transform import from_origin
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# ============================================================
# 1. CONFIGURATION (self-contained)
# ============================================================

# Output directory
SOIL_DIR = "./Data/Datasets/ground_truth-9CH/Raw/Soil"
os.makedirs(SOIL_DIR, exist_ok=True)

# Your bounding boxes (min_lon, min_lat, max_lon, max_lat)
TP_BOUNDING_BOXES = [
                        (-1.875,51.145,-1.775,51.205),
                        (-1.895,51.405,-1.815,51.455),
                        (-1.895,51.31,-1.85,51.338),
                        (-1.12,51.335,-1.05,51.38),
                        (1.255,52.565,1.325,52.6),
                        (1.275,52.608,1.322,52.635),
                        (-2.445,50.7,-2.407,50.722),
                        (1.246,52.565,1.293,52.592),
                        (-1.82,51.165,-1.72,51.225),
                        (-1.84,51.425,-1.76,51.475),
                        (-1.84,51.33,-1.795,51.358),
                        (-1.688,51.098,-1.644,51.128),
                        (-1.065,51.355,-0.995,51.4),
                        (1.31,52.585,1.38,52.62),
                        (1.33,52.628,1.377,52.655),
                        (-2.39,50.72,-2.352,50.742),
                        (1.301,52.585,1.348,52.612),

]

# SoilGrids layers
SOIL_LAYERS = ["bdod", "clay", "sand", "silt", "soc"]
DEPTH = "0-5cm"

# SoilGrids API
BASE_URL = "https://rest.isric.org/soilgrids/v2.0/properties/query"

# SoilGrids native resolution (~250m)
RES = 0.00225


# ============================================================
# 2. Query a single point
# ============================================================
def query_point(lon, lat, layer, depth, retries=3):
    params = {
        "property": layer,
        "depth": depth,
        "lat": lat,
        "lon": lon,
        "format": "json"
    }

    for attempt in range(retries):
        try:
            r = requests.get(BASE_URL, params=params, timeout=10)
            if r.status_code != 200:
                return np.nan
            data = r.json()
            return data["properties"][layer]["value"]
        except Exception:
            time.sleep(0.5 * (attempt + 1))

    return np.nan


# ============================================================
# 3. Build raster for one layer + one bounding box
# ============================================================
def build_raster_for_layer(layer, bbox, batch_idx):
    min_lon, min_lat, max_lon, max_lat = bbox

    # Create tile directory
    tile_dir = os.path.join(SOIL_DIR, f"tile {batch_idx}")
    os.makedirs(tile_dir, exist_ok=True)

    # Build grid coordinates
    lons = np.arange(min_lon, max_lon, RES)
    lats = np.arange(max_lat, min_lat, -RES)

    grid = np.zeros((len(lats), len(lons)), dtype="float32")

    # Prepare tasks
    tasks = []
    with ThreadPoolExecutor(max_workers=20) as executor:
        for i, lat in enumerate(lats):
            for j, lon in enumerate(lons):
                tasks.append(executor.submit(query_point, lon, lat, layer, DEPTH))

        # Fill grid with results
        idx = 0
        for future in tqdm(as_completed(tasks), total=len(tasks), desc=f"{layer} (tile {batch_idx})"):
            value = future.result()
            i = idx // len(lons)
            j = idx % len(lons)
            grid[i, j] = value
            idx += 1

    # Build transform
    transform = from_origin(lons.min(), lats.max(), RES, RES)

    out_path = os.path.join(tile_dir, f"tile_{batch_idx}_{layer}.tif")

    profile = {
                "driver": "GTiff",
                "height": grid.shape[0],
                "width": grid.shape[1],
                "count": 1,
                "dtype": "float32",
                "crs": "EPSG:4326",
                "transform": transform
    }

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(grid, 1)

    print(f"[OK] Created {out_path}")


# ============================================================
# 4. MAIN LOOP — iterate over bounding boxes + layers
# ============================================================
for batch_idx, bbox in enumerate(TP_BOUNDING_BOXES):
    print(f"\n=== Processing bounding box {batch_idx}: {bbox} ===")

    for layer in SOIL_LAYERS:
        print(f"[BUILD] {layer} for tile {batch_idx}")
        build_raster_for_layer(layer, bbox, batch_idx)



=== Processing bounding box 0: (-1.875, 51.145, -1.775, 51.205) ===
[BUILD] bdod for tile 0


bdod (tile 0): 100%|██████████| 1215/1215 [00:17<00:00, 68.85it/s] 


[OK] Created ./Data/Datasets/ground_truth-9CH/Raw/Soil\tile 0\tile_0_bdod.tif
[BUILD] clay for tile 0


clay (tile 0): 100%|██████████| 1215/1215 [00:12<00:00, 97.95it/s] 


[OK] Created ./Data/Datasets/ground_truth-9CH/Raw/Soil\tile 0\tile_0_clay.tif
[BUILD] sand for tile 0


...

[OK] Created ./Data/Datasets/ground_truth-9CH/Raw/Soil\tile 16\tile_16_sand.tif
[BUILD] silt for tile 16


silt (tile 16): 100%|██████████| 273/273 [00:04<00:00, 61.36it/s] 


[OK] Created ./Data/Datasets/ground_truth-9CH/Raw/Soil\tile 16\tile_16_silt.tif
[BUILD] soc for tile 16


soc (tile 16): 100%|██████████| 273/273 [00:04<00:00, 57.71it/s] 

[OK] Created ./Data/Datasets/ground_truth-9CH/Raw/Soil\tile 16\tile_16_soc.tif
